# SehatSamjho — Day 1 Extraction Test

Tests GPT-4o Vision extraction directly without WhatsApp/Twilio.

**Run from:** `SehatSamjho/` root  
**Requires:** `.env` with valid `OPENAI_API_KEY`

**Image options:**
- Drop a prescription photo into `data/prescriptions/` and use `Path` to load it
- Or pass any public image URL directly

In [1]:
# ── Setup: load .env and add backend/ to path ─────────────────────────────────
import sys
from pathlib import Path

# Notebook lives in notebooks/, backend/ is one level up
REPO_ROOT = Path().resolve().parent
BACKEND   = REPO_ROOT / "backend"
DATA_DIR  = REPO_ROOT / "data" / "prescriptions"

if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

print(f"Backend path : {BACKEND}")
print(f"Images folder: {DATA_DIR}")
print(f"Images found : {[f.name for f in DATA_DIR.glob('*') if f.is_file()]}")

Backend path : /Users/nishantgaurav/Project/SehatSamjho/backend
Images folder: /Users/nishantgaurav/Project/SehatSamjho/data/prescriptions
Images found : ['10.jpg']


In [2]:
# ── Verify config loaded ──────────────────────────────────────────────────────
from app.core.config import settings

print(f"OpenAI key : {'SET ✓' if settings.openai_api_key.startswith('sk-') else 'MISSING ✗'}")
print(f"Model      : {settings.openai_model}")
print(f"Dry run    : {settings.twilio_dry_run}")

OpenAI key : SET ✓
Model      : gpt-4o
Dry run    : True


## Option A — Test with a local image file

1. Drop your prescription photo into `data/prescriptions/`
2. Update the filename below

> The image must be uploaded to a public URL for GPT-4o Vision to read it.
> Cell below uses a free [imgbb](https://imgbb.com) upload — or paste any direct image URL you already have.

In [9]:
# Upload a local file and get a public URL for GPT-4o
# (GPT-4o Vision requires a URL, not a local path)
import base64, httpx

IMAGE_FILE = DATA_DIR / "101.jpg"   # ← change filename here

if IMAGE_FILE.exists():
    # Encode as base64 data URL — GPT-4o supports this directly
    img_bytes = IMAGE_FILE.read_bytes()
    b64 = base64.b64encode(img_bytes).decode()
    ext = IMAGE_FILE.suffix.lstrip(".").lower()
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    IMAGE_URL = f"data:{mime};base64,{b64}"
    print(f"Loaded {IMAGE_FILE.name} ({len(img_bytes)/1024:.1f} KB) — using base64 data URL")
else:
    print(f"File not found: {IMAGE_FILE}")
    print(f"Files in {DATA_DIR}: {[f.name for f in DATA_DIR.glob('*')]}")

Loaded 101.jpg (87.7 KB) — using base64 data URL


## Option B — Test with a public URL

Paste any publicly accessible prescription image URL here.

In [ ]:
# Paste a public image URL directly (skip Option A if using this)
# IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/7/7d/Prescription_handwritten.jpg/800px-Prescription_handwritten.jpg"
# print(f"Using URL: {IMAGE_URL}")

## Run Extraction

In [10]:
import asyncio
from app.services.extraction import extract_prescription

print("Calling GPT-4o Vision...\n")

prescription, confidence = await extract_prescription(IMAGE_URL, request_id="nb-test-001")

print(f"✅ Extraction complete!")
print(f"   Doc type   : {prescription.doc_type.value}")
print(f"   Confidence : {confidence:.2f}")
print(f"   Medicines  : {len(prescription.medications)}")
print(f"   Diagnosis  : {prescription.diagnosis or '—'}")
print(f"   Doctor     : {prescription.doctor_name or '—'}")
print(f"   Date       : {prescription.date or '—'}")

2026-02-27 23:12:55.443 | INFO     | app.services.extraction:extract_prescription:109 - Extracting prescription from image: data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGB...
2026-02-27 23:12:55.445 | DEBUG    | app.services.extraction:_call_openai_with_retry:146 - Calling GPT-4o Vision API


Calling GPT-4o Vision...



2026-02-27 23:13:08.275 | INFO     | app.services.extraction:extract_prescription:123 - Extraction complete: 3 meds, doc_type=prescription, confidence=0.56


✅ Extraction complete!
   Doc type   : prescription
   Confidence : 0.56
   Medicines  : 3
   Diagnosis  : —
   Doctor     : —
   Date       : —


In [11]:
# ── Medicines detail ──────────────────────────────────────────────────────────
print(f"💊 Medicines ({len(prescription.medications)}):")
for i, med in enumerate(prescription.medications, 1):
    print(f"  {i}. {med.name}")
    if med.dosage:        print(f"     Dose      : {med.dosage}")
    if med.frequency:     print(f"     Frequency : {med.frequency}")
    if med.duration:      print(f"     Duration  : {med.duration}")
    if med.instructions:  print(f"     Notes     : {med.instructions}")

💊 Medicines (3):
  1. T. Dicoson
     Duration  : 8 days
  2. T. Rantaen
  3. T. BL


In [12]:
# ── Warnings & instructions ───────────────────────────────────────────────────
if prescription.warnings:
    print("⚠️  Warnings:")
    for w in prescription.warnings:
        print(f"   • {w}")

if prescription.general_instructions:
    print(f"\n📝 Instructions: {prescription.general_instructions}")

print(f"\n📄 Raw text (first 300 chars):\n{prescription.raw_text[:300]}")


📄 Raw text (first 300 chars):
Medicines Prescribed:
S.no | Medicine | Strength | Days | Timings | Morning Afternoon Evening
(CAPITAL LETTERS PLEASE)

1 T Dicoson 8days
2 T Rantaen
3 T. BL

Follow up advice:
Doctor: 

[Signature] 8/5/21


In [13]:
# ── Preview the WhatsApp reply that would be sent ─────────────────────────────
import sys
sys.path.insert(0, str(BACKEND))
from app.api.webhooks import _format_extraction_reply

reply = _format_extraction_reply(prescription, language_name="Hindi")
print("📱 WhatsApp message that would be sent:")
print("─" * 50)
print(reply)

📱 WhatsApp message that would be sent:
──────────────────────────────────────────────────
📋 *Document Summary*

*Type:* Prescription

💊 *Medicines (3):*
  • T. Dicoson — for 8 days
  • T. Rantaen
  • T. BL

---
_⚕️ This is an automated extraction. Always follow your doctor's advice._
_Note: Full translation in Hindi coming in next update._


## Test the full state machine via curl (optional)

Run these in a terminal to walk through the conversation flow:

In [8]:
# Print ready-to-run curl commands
import urllib.parse

encoded_url = urllib.parse.quote(IMAGE_URL, safe='')

print("# Step 1 — New user (gets language menu)")
print('curl -X POST http://localhost:8000/webhook/whatsapp \\')
print('  -d "From=whatsapp%3A%2B919876543210&To=whatsapp%3A%2B14155238886&Body=Start&NumMedia=0&MessageSid=SM001"')
print()
print("# Step 2 — Select Hindi")
print('curl -X POST http://localhost:8000/webhook/whatsapp \\')
print('  -d "From=whatsapp%3A%2B919876543210&To=whatsapp%3A%2B14155238886&Body=1&NumMedia=0&MessageSid=SM002"')
print()
print("# Step 3 — Send image")
print(f'curl -X POST http://localhost:8000/webhook/whatsapp \\')
print(f'  -d "From=whatsapp%3A%2B919876543210&To=whatsapp%3A%2B14155238886&Body=&NumMedia=1&MediaUrl0={encoded_url}&MediaContentType0=image%2Fjpeg&MessageSid=SM003"')

# Step 1 — New user (gets language menu)
curl -X POST http://localhost:8000/webhook/whatsapp \
  -d "From=whatsapp%3A%2B919876543210&To=whatsapp%3A%2B14155238886&Body=Start&NumMedia=0&MessageSid=SM001"

# Step 2 — Select Hindi
curl -X POST http://localhost:8000/webhook/whatsapp \
  -d "From=whatsapp%3A%2B919876543210&To=whatsapp%3A%2B14155238886&Body=1&NumMedia=0&MessageSid=SM002"

# Step 3 — Send image
curl -X POST http://localhost:8000/webhook/whatsapp \
  -d "From=whatsapp%3A%2B919876543210&To=whatsapp%3A%2B14155238886&Body=&NumMedia=1&MediaUrl0=data%3Aimage%2Fjpeg%3Bbase64%2C%2F9j%2F4AAQSkZJRgABAQAAAQABAAD%2F2wBDABQODxIPDRQSEBIXFRQYHjIhHhwcHj0sLiQySUBMS0dARkVQWnNiUFVtVkVGZIhlbXd7gYKBTmCNl4x9lnN%2BgXz%2F2wBDARUXFx4aHjshITt8U0ZTfHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHx8fHz%2FwgARCAUAA8ADASIAAhEBAxEB%2F8QAGQABAQEBAQEAAAAAAAAAAAAAAAECAwQF%2F8QAGAEBAQEBAQAAAAAAAAAAAAAAAAECAwT%2F2gAMAwEAAhADEAAAAeDNzRSJUwUSis1dSUqUrOiEiTcM2UTcEsJqaKmTWWS6zCpoLCxk0zRZDpM0Z1kytC0xnrzM